# Plotting Subgoal Proposer

In [12]:
# PATH = '/home/jennifer/scratch/aorl2/2026-03-16-00/2026-03-16-00.682e4de5697b9c301a2fa322d299ef547e570517f079b5019aacc5dfc5697c07/'
PATH = '../../scratch/aorl2/2026-03-16-00/2026-03-16-00.0dfb77fdff1e9bb91c8f7575c8ad2c3f7d867f93572cfb2780acc46d86598233/'
CKPT_NUM = 1000000

In [13]:
import glob
import json
import os
import pathlib
import random
import signal
import sys
import time
from collections import defaultdict

import jax
import numpy as np
import tqdm
import wandb
import wandb.util
from absl import app, flags
from ml_collections import config_flags

from agents import agents
from utils.datasets import Dataset, GCDataset, HGCDataset, ReplayBuffer
from utils.flax_utils import restore_agent, save_agent
from utils.log_utils import CsvLogger, get_animal, get_exp_name, get_flag_dict, setup_wandb
from utils.plot_utils import plot_heatmap
from utils.samplers import to_oracle_rep
from utils.statistics import get_statistics_class
from wrappers.datafuncs_utils import clip_dataset, make_env_and_datasets
from utils.evaluation import evaluate_gcfql, evaluate
from ogbench.relabel_utils import add_oracle_reps, relabel_dataset

# Simplified setup - manually set the key flags
# FLAGS = flags.FLAGS

In [14]:
flags_file = os.path.join(PATH, 'flags.json')
with open(flags_file, 'r') as f:
    saved_flags = json.load(f)

print(saved_flags.keys())

dict_keys(['logtostderr', 'alsologtostderr', 'log_dir', 'v', 'verbosity', 'logger_levels', 'stderrthreshold', 'showprefixforinfo', 'run_with_pdb', 'pdb_post_mortem', 'pdb', 'run_with_profiling', 'profile_file', 'use_cprofile_for_profiling', 'only_check_args', 'chex_n_cpu_devices', 'chex_assert_multiple_cpu_devices', 'test_srcdir', 'test_tmpdir', 'test_random_seed', 'test_randomize_ordering_seed', 'xml_output_file', 'chex_skip_pmap_variant_if_single_device', 'pymjcf_debug', 'pymjcf_debug_full_dump_dir', 'pymjcf_log_xml', 'timeout', 'wbproj', 'run_group', 'seed', 'wandb_alerts', 'env_name', 'dataset_dir', 'agent', 'fql_agent', 'save_dir', 'offline_steps', 'further_offline_steps', 'log_interval', 'eval_interval', 'save_interval', 'collection_steps', 'data_plot_interval', 'cleanup', 'steps_toward_sg', 'num_subgoal_candidates', 'use_triangle', 'triangle_threshold', 'subgoal_reached_distance', 'eval_episodes', 'eval_temperature', 'eval_gaussian', 'video_episodes', 'video_frame_skip', '?', 'h

In [16]:
# Print FLAGS in a readable format
print("Current FLAGS:")
for flag_name in sorted(dir(saved_flags)):
    if not flag_name.startswith('_'):
        try:
            value = getattr(saved_flags, flag_name)
            if not callable(value):
                print(f"  {flag_name}: {value}")
        except:
            pass

Current FLAGS:


In [21]:
agent_config = saved_flags['agent']
print(agent_config.keys())

dict_keys(['action_chunking', 'action_dim', 'actor_geom_sample', 'actor_hidden_dims', 'actor_p_curgoal', 'actor_p_randomgoal', 'actor_p_trajgoal', 'agent_name', 'alpha', 'awr_invtemp', 'batch_size', 'best_of_n', 'critic_loss_type', 'dataset_class', 'discount', 'flow_steps', 'gc_negative', 'goal_proposer_type', 'horizon_length', 'layer_norm', 'lr', 'num_qs', 'ob_dims', 'q_agg', 'subgoal_steps', 'tau', 'train_goal_proposer', 'train_value', 'value_geom_sample', 'value_hidden_dims', 'value_p_curgoal', 'value_p_randomgoal', 'value_p_trajgoal'])


In [19]:
train_dataset = np.load(os.path.join(PATH, 'data-100000.npz'))

In [22]:
from utils.datasets import GCDataset

train_dataset = GCDataset(Dataset.create(**train_dataset), config=agent_config)

In [23]:
example_batch = train_dataset.sample(1)
example_batch.keys()

dict_keys(['actions', 'next_observations', 'observations', 'oracle_reps', 'qpos', 'qvel', 'terminals', 'value_goals', 'actor_goals', 'masks', 'rewards', 'low_actor_goals'])

In [25]:
# Use saved_flags directly instead of FLAGS
# env_name = saved_flags['env_name']
# dataset_dir = saved_flags.get('dataset_dir', '/home/jennifer/scratch/data/humanoidmaze-medium-navigate-v0')
# agent_config = saved_flags.get('agent', {
#     'agent_name': 'gcfql',
#     'train_goal_proposer': True,
#     'goal_proposer_type': 'low_actor_goals',
#     'train_value': True,
#     'horizon_length': 25,
#     'action_chunking': False,
#     'num_qs': 10,
#     'q_agg': 'mean',
#     'discount': 0.995,
#     'batch_size': 256,
#     'alpha': 30.0,
#     'subgoal_steps': 25,
#     'awr_invtemp': 0.0,
# })
seed = saved_flags.get('seed', 0)

# Create environment
# datasets = [file for file in sorted(glob.glob(f'{saved_flags["dataset_dir"]}/*.npz')) if '-val.npz' not in file]
# dataset_idx = 0
# splits = saved_flags['env_name'].split('-')
# env_name = '-'.join(splits[:-3]) + '-v0'
# env, train_dataset_data, val_dataset_data = make_env_and_datasets(
#     env_name,
#     dataset_path=datasets[dataset_idx],
#     use_oracle_reps=True,
# )

# Get a sample observation and goal
# train_dataset = GCDataset(Dataset.create(**train_dataset_data, freeze=False), agent_config)
example_batch = train_dataset.sample(1)
observation = example_batch['observations'][0]
# goal = example_batch['goals'][0]

# Restore agent
# agent_config = FLAGS.agent
agent = agents[agent_config['agent_name']].create(seed, example_batch, agent_config)
agent = restore_agent(agent, PATH, CKPT_NUM)

Restored from ../../scratch/aorl2/2026-03-16-00/2026-03-16-00.0dfb77fdff1e9bb91c8f7575c8ad2c3f7d867f93572cfb2780acc46d86598233//params_1000000.pkl


```
{'task_name': 'task4', 'init_ij': (6, 5), 'init_xy': (16.0, 20.0), 'goal_ij': (6, 1), 'goal_xy': (0.0, 20.0)}
```

In [26]:
goal = np.array([0.0, 20.0])

In [27]:
# Sample 128 subgoals
rng = jax.random.PRNGKey(seed)
observations = np.repeat(observation[None], 128, axis=0)
goals = np.repeat(goal[None], 128, axis=0)
subgoals = np.asarray(agent.propose_goals(observations, goals, rng))

print(f"Sampled {len(subgoals)} subgoals")
print(f"Observation: {observation}")
print(f"Goal: {goal}")
print(f"First few subgoals: {subgoals[:5]}")

Sampled 128 subgoals
Observation: [ 1.0797711e+01  1.2035127e+01 -1.6328716e-01 -5.3082603e-01
  6.4935291e-01 -2.7710682e-02 -1.1064184e+00 -4.3954125e-01
 -3.1053063e-01 -7.8616595e-01  9.2785549e-01  9.5872939e-02
 -1.6614679e-01 -2.2248665e-02  4.9340036e-02  1.0059745e+00
  1.0950311e+00  1.1051751e+00  1.0147769e+00 -1.6631020e+00
 -1.0031685e+00 -1.8773292e-01  1.0784057e+00  1.3634930e+00
  3.6915001e-02  1.5223278e-01  8.4069148e-02  4.6565834e-01
  4.9923468e-01 -8.9827675e-01  5.2594686e-01  1.5596333e-01
  5.8405396e-02  6.6729617e-01  1.4543623e-01 -9.8576725e-01
 -1.6740561e-01 -3.4173301e-01  9.2476696e-01 -8.7674953e-02
 -1.9192159e+00 -5.1503056e-01  3.1253493e-01 -2.1793485e+00
 -8.1193602e-01  3.2160494e+00 -1.8732051e+00  9.1516197e-01
 -5.3598361e+00  3.1570801e-01  2.3101695e+00 -5.5612669e+00
 -4.8394758e-01 -1.0004672e+01 -2.2642773e+01 -3.9033016e+01
 -1.3320187e+00  3.1064281e-01  5.3530622e+00  9.8664217e+00
  5.8650023e-01 -4.4831572e+00 -5.6703315e+00 -1.48

In [28]:
subgoals

array([[ 9.9670514e-04,  2.0755249e+01],
       [-3.2521751e-02,  2.0786182e+01],
       [-1.2765246e-02,  2.0692486e+01],
       [-1.1949388e-02,  2.0777060e+01],
       [-3.5166647e-02,  2.0667391e+01],
       [-7.7540651e-02,  2.0734140e+01],
       [-8.0618560e-03,  2.0730303e+01],
       [-6.2144302e-02,  2.0798189e+01],
       [ 7.0562996e-03,  2.0656803e+01],
       [-9.3203962e-02,  2.0699436e+01],
       [ 2.0301193e-02,  2.0742262e+01],
       [ 4.9369782e-03,  2.0744816e+01],
       [-3.2827277e-02,  2.0735022e+01],
       [-1.1977056e-02,  2.0663734e+01],
       [-1.9272914e-02,  2.0773752e+01],
       [-4.1313969e-02,  2.0698627e+01],
       [-1.0114831e-01,  2.0859999e+01],
       [-7.6946929e-02,  2.0814617e+01],
       [-6.9705471e-03,  2.0704702e+01],
       [ 9.1181099e-03,  2.0745874e+01],
       [-1.4623309e-02,  2.0795458e+01],
       [ 3.8949057e-02,  2.0610832e+01],
       [-3.3283636e-02,  2.0756197e+01],
       [-2.2135802e-02,  2.0663542e+01],
       [-1.39261